# Tool catalog: processes and terminals

Run two bounded subprocesses, then create one disposable tmux pane to inspect. Setup uses a private tmux socket directory, so the examples do not list or read your normal terminal sessions. Only this notebook-created pane is read. Cleanup removes it.


In [ ]:
from nbinlineai.tools import run_python, run_shell, tmux_sessions, tmux_read
from tempfile import TemporaryDirectory
import os, subprocess, uuid
catalog_tmux_temp = TemporaryDirectory(prefix="nbinlineai-tmux-")
catalog_prior_tmux_dir = os.environ.get("TMUX_TMPDIR")
os.environ["TMUX_TMPDIR"] = catalog_tmux_temp.name
catalog_tmux_name = "nbinlineai_catalog_" + uuid.uuid4().hex[:10]
subprocess.run(["tmux", "new-session", "-d", "-s", catalog_tmux_name], check=True)
catalog_pane_id = subprocess.check_output(["tmux", "list-panes", "-t", catalog_tmux_name, "-F", "#{pane_id}"], text=True).strip()
subprocess.run(["tmux", "send-keys", "-t", catalog_pane_id, "printf catalog-pane-ready", "Enter"], check=True)
print("Owned disposable pane:", catalog_pane_id)


### Offered tools

- &`run_python`
- &`run_shell`
- &`tmux_sessions`
- &`tmux_read`


## run_python


In [ ]:
print(run_python("print(2 + 3)", timeout=5))


## run_shell


In [ ]:
print(run_shell("echo catalog-ready", timeout=5))


## Ask AI to run and inspect the same small tasks

The Python and shell tools run subprocesses; the tmux tools inspect only the owned pane from setup.


Use &`run_python` for `print(2 + 3)` and &`run_shell` for `echo catalog-ready`, both with a five-second timeout. State each actual exit status and output.


## tmux_sessions


In [ ]:
print(tmux_sessions())


Use &`tmux_sessions` to find the owned pane $`catalog_pane_id`. State its session and pane ID; ignore other panes.


## tmux_read


In [ ]:
print(tmux_read(catalog_pane_id, lines=10))


Use &`tmux_read` with pane $`catalog_pane_id` and `lines=10`. Report whether its screen contains `catalog-pane-ready`. Do not send input.


In [ ]:
subprocess.run(["tmux", "kill-session", "-t", catalog_tmux_name], check=True)
if catalog_prior_tmux_dir is None:
    os.environ.pop("TMUX_TMPDIR", None)
else:
    os.environ["TMUX_TMPDIR"] = catalog_prior_tmux_dir
catalog_tmux_temp.cleanup()
print("Owned tmux session removed.")
